# Gate 2 - Part 3/3: read the verdict (CPU - no GPU needed)

**Upload the `pop_sweep_ne<BEST_NE>.csv` from Part 2** in section 1. Prints the paired berry-1 norm signal `d1 = eat1[(0,)] - eat1[(0,1)]` per N (SURVIVES at margin>=2) and the compliance regime check. The lowest surviving N is the population the extinction experiment runs at.

Pure analysis - runs on a CPU runtime. No source files or GPU required.

## 1. Upload the pilot CSV from Part 2

In [ ]:
from google.colab import files
up = files.upload()
CSV = list(up)[0]; print('analyzing', CSV)

## 2. Verdict

In [ ]:
import csv as _csv, numpy as np

def tail_cells(path, tail_frac=0.1):
    rows = list(_csv.DictReader(open(path)))
    umax = max(int(r['update']) for r in rows); cut = umax - int(tail_frac * (umax + 1))
    acc = {}
    for r in rows:
        if int(r['update']) < cut: continue
        k = (int(r['N']), r['condition'], int(r['seed']))
        acc.setdefault(k, []).append((float(r['eat0']), float(r['eat1'])))
    cells = {}
    for (N, cond, s), vals in acc.items():
        cells.setdefault(N, {}).setdefault(cond, {})[s] = np.array(vals).mean(0)
    return cells

def paired(a, b):
    seeds = sorted(set(a) & set(b)); d = np.array([a[s] - b[s] for s in seeds])
    if len(d) < 2: return (float(d.mean()) if len(d) else float('nan')), float('nan'), len(d)
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d))), len(d)

cells = tail_cells(CSV)
print('berry-1 norm signal d1 = eat1[(0,)] - eat1[(0,1)], paired; SURVIVES if margin>=2\n')
print(f"{'N':>3} | {'eat1 (0,)':>10} {'eat1 (0,1)':>11} | {'d1':>7} {'SEM':>6} {'margin':>7} | verdict")
print('-' * 72)
for N in sorted(cells):
    c = cells[N]
    if '0' not in c or '01' not in c:
        print(f'{N:>3} | incomplete'); continue
    a = {s: v[1] for s, v in c['0'].items()}; b = {s: v[1] for s, v in c['01'].items()}
    m, sem, n = paired(a, b)
    margin = m / sem if sem and np.isfinite(sem) and sem > 0 else float('nan')
    surv = 'SURVIVES' if (np.isfinite(margin) and margin >= 2) else 'weak/none'
    print(f'{N:>3} | {np.mean(list(a.values())):>10.1f} {np.mean(list(b.values())):>11.1f} | '
          f'{m:>7.2f} {sem:>6.2f} {margin:>7.2f} | {surv}')
print('\ncompliance regime check (koster = none ~90 >> (0,1) ~45):')
print(f"{'N':>3} | {'none':>7} {'(0,)':>7} {'(0,1)':>7}")
for N in sorted(cells):
    c = cells[N]
    g = lambda cc: np.mean([v[0] for v in c[cc].values()]) if cc in c else float('nan')
    print(f'{N:>3} | {g("none"):>7.1f} {g("0"):>7.1f} {g("01"):>7.1f}')